# Ride-Sharing Demand & Revenue Intelligence
## NYC Yellow Taxi — August 2026
### Notebook 01 — Data Cleaning & Feature Engineering

**Business objective:** Prepare August 2026 NYC TLC Yellow Taxi trip records for demand, recorded-charge, geographic and fare-prediction analysis.

**Business / data-quality questions**
1. How many raw records exist, and how many pickups actually occurred in August 2026?
2. Which attributes are incomplete? Are missing values clustered together?
3. How many records have invalid pickup/drop-off times, distances, fares or totals?
4. Are extreme fares, distances and durations possible genuine trips or errors?
5. Which engineered features are needed for the next notebooks?
6. How many records are retained, and how does cleaning affect the total recorded amount?

**Note:** No January 2025 numbers are copied here. Run every cell on the actual August 2026 file.

## 1. Import libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries imported")

## 2. Dataset loading

Store the file at `Ride-Sharing-Demand-Intelligence/Raw/yellow_tripdata_2026-08.parquet`, and launch Jupyter from the project folder. This path code supports notebooks running from either the project root or `Notebooks/`.

In [ ]:
project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

file_path = project_root / "Raw" / "yellow_tripdata_2026-08.parquet"
output_dir = project_root / "Processed"
output_dir.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Dataset found:", file_path.exists())
assert file_path.exists(), f"Dataset not found: {file_path}"

df = pd.read_parquet(file_path)
display(df.head())

## 3. Raw data exploration

Understand the number of records, available columns, data types and memory footprint *before* removing anything.

In [ ]:
print("Raw dataset shape:", df.shape)
df.info(show_counts=True)
print("Columns:", df.columns.tolist())

## 4. Missing-value analysis

Missing is not the same as zero. Investigate nulls before choosing whether to drop, retain or impute.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2)
}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Columns containing nulls:", df.columns[df.isna().any()].tolist())

In [ ]:
# Are the same rows missing multiple reported attributes?
possible_cluster = [
    "passenger_count", "RatecodeID", "store_and_fwd_flag",
    "congestion_surcharge", "Airport_fee"
]
cluster = [c for c in possible_cluster if c in df.columns]
if cluster:
    missing_pattern = (
        df[cluster].isna().value_counts(dropna=False).head(10)
        .rename("rows").reset_index()
    )
    display(missing_pattern)

## 5. Select analysis columns

The January 2025 reference immediately dropped `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee`. Here, **we do not discard fee/rate fields simply because they contain nulls**. We retain the available fields, and the next analysis notebook can select just what it needs. This also accommodates schema changes between years.

In [ ]:
columns_to_keep = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee"
]
existing = [c for c in columns_to_keep if c in df.columns]
missing_expected = [c for c in columns_to_keep if c not in df.columns]
print("Expected columns absent from 2026 file:", missing_expected)
df_1 = df[existing].copy()
print("Working dataset:", df_1.shape)
display(df_1.head())

## 6. Investigate invalid or suspicious records

These checks are **diagnostics**, not deletion instructions. Zero/negative fares may reflect reversals or adjustments; missing passengers do not necessarily imply zero passengers.

In [ ]:
required = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "total_amount"]
absent = [c for c in required if c not in df_1]
assert not absent, f"Required fields missing: {absent}"

for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    df_1[c] = pd.to_datetime(df_1[c], errors="coerce")

aug_start = pd.Timestamp("2026-08-01")
sep_start = pd.Timestamp("2026-09-01")
duration_minutes = (df_1["tpep_dropoff_datetime"] - df_1["tpep_pickup_datetime"]).dt.total_seconds() / 60

checks = {
    "pickup missing": df_1["tpep_pickup_datetime"].isna(),
    "dropoff missing": df_1["tpep_dropoff_datetime"].isna(),
    "pickup outside Aug 2026": ~df_1["tpep_pickup_datetime"].between(aug_start, sep_start, inclusive="left"),
    "nonpositive duration": duration_minutes <= 0,
    "distance missing/nonpositive": df_1["trip_distance"].isna() | (df_1["trip_distance"] <= 0),
    "fare missing/nonpositive": df_1["fare_amount"].isna() | (df_1["fare_amount"] <= 0),
    "total missing/nonpositive": df_1["total_amount"].isna() | (df_1["total_amount"] <= 0),
}
if "passenger_count" in df_1:
    checks["passenger count missing"] = df_1["passenger_count"].isna()
    checks["passenger count nonpositive"] = df_1["passenger_count"] <= 0

quality_check = pd.Series({name: int(mask.sum()) for name, mask in checks.items()}, name="row_count")
display(quality_check.to_frame())
print("Fully identical raw rows:", int(df_1.duplicated().sum()))
print("Note: counts may overlap; do not sum them to estimate rows excluded.")

## 7. Apply documented cleaning rules

**Chosen definition for the main completed-trip analytics dataset:** August pickup, valid pickup/dropoff timestamps, positive duration, distance, metered fare and recorded total. We retain missing passenger counts as `NaN` for honest reporting. Fee adjustments and excluded records remain in the original file. We do **not** claim these are the only valid TLC records.

For auditability, compare rows and `total_amount` before/after cleaning. These filters can exclude some legitimate adjustments; document that limitation.

In [ ]:
valid_mask = (
    df_1["tpep_pickup_datetime"].ge(aug_start)
    & df_1["tpep_pickup_datetime"].lt(sep_start)
    & df_1["tpep_pickup_datetime"].notna()
    & df_1["tpep_dropoff_datetime"].notna()
    & duration_minutes.gt(0)
    & df_1["trip_distance"].gt(0)
    & df_1["fare_amount"].gt(0)
    & df_1["total_amount"].gt(0)
)

before_count = len(df_1)
before_total = df_1["total_amount"].sum()
clean_df = df_1.loc[valid_mask].copy()
print("Raw rows:", before_count)
print("Retained rows:", len(clean_df))
print("Excluded rows:", before_count - len(clean_df))
print("Percent retained:", round(100 * len(clean_df) / before_count, 2))
print("Raw sum(total_amount):", round(before_total, 2))
print("Clean sum(total_amount):", round(clean_df["total_amount"].sum(), 2))

## 8. Feature engineering

Create features used in the EDA and regression notebooks.

- `trip_duration_min`: elapsed time in minutes
- `pickup_hour`, `pickup_day`, `pickup_weekday`, `is_weekend`: demand groupings
- `revenue_per_mile`, `revenue_per_minute`: recorded-charge ratios (not operator profit)
- `tip_percentage`: ratio, where 0.20 = 20%; the raw `tip_amount` does not capture cash tips reliably

In [ ]:
clean_df["trip_duration_min"] = (
    (clean_df["tpep_dropoff_datetime"] - clean_df["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
).round(2)

clean_df["pickup_hour"] = clean_df["tpep_pickup_datetime"].dt.hour
clean_df["pickup_day"] = clean_df["tpep_pickup_datetime"].dt.day
clean_df["pickup_weekday"] = clean_df["tpep_pickup_datetime"].dt.weekday
clean_df["is_weekend"] = clean_df["pickup_weekday"].isin([5, 6]).astype("int8")

clean_df["revenue_per_mile"] = (clean_df["total_amount"] / clean_df["trip_distance"]).round(2)
clean_df["revenue_per_minute"] = (clean_df["total_amount"] / clean_df["trip_duration_min"]).round(2)
clean_df["tip_percentage"] = (clean_df["tip_amount"] / clean_df["fare_amount"]).round(4) if "tip_amount" in clean_df else np.nan

display(clean_df.head())

## 9. Outlier investigation — do not delete automatically

The January reference used the 1st and 99th percentiles to remove trip fares and distances. We will first **inspect** those thresholds, including duration and recorded total, rather than assume every extreme trip is an error. Airport or long-distance rides can be genuine. The main `clean_df` is kept unchanged.

In [ ]:
outlier_columns = ["trip_distance", "fare_amount", "total_amount", "trip_duration_min"]
quantiles = clean_df[outlier_columns].quantile([0.01, 0.50, 0.95, 0.99, 0.999]).T
quantiles.columns = ["p01", "median", "p95", "p99", "p99.9"]
display(quantiles)

for column in outlier_columns:
    threshold = quantiles.loc[column, "p99"]
    print(f"{column}: {int((clean_df[column] > threshold).sum()):,} records above p99 ({threshold:,.2f})")

In [ ]:
# Optional comparison ONLY: count records under old 1%-99% trimming rule.
# This does not overwrite clean_df or silently remove genuine long trips.
fare_lo, fare_hi = clean_df["fare_amount"].quantile([0.01, 0.99])
dist_lo, dist_hi = clean_df["trip_distance"].quantile([0.01, 0.99])
trimmed_comparison = clean_df.loc[
    clean_df["fare_amount"].between(fare_lo, fare_hi)
    & clean_df["trip_distance"].between(dist_lo, dist_hi)
]
print("Main cleaned rows:", len(clean_df))
print("Rows after optional 1%-99% trim:", len(trimmed_comparison))
print("Rows removed by optional trim:", len(clean_df) - len(trimmed_comparison))

## 10. Sanity checks

Check that feature values and date range match expectations. Extremely large duration or charge ratios should be investigated before making business claims or training machine learning models.

In [ ]:
display(clean_df.describe(include="all").T)
print("Date range:", clean_df["tpep_pickup_datetime"].min(), "to", clean_df["tpep_pickup_datetime"].max())
print("Cleaned shape:", clean_df.shape)
assert clean_df["pickup_hour"].between(0, 23).all()
assert clean_df["is_weekend"].isin([0, 1]).all()
assert clean_df["trip_duration_min"].gt(0).all()

## 11. Save cleaned dataset

Save Parquet for efficient SQL/ML use and CSV only if required for Power BI. The files go into your local `Processed/` folder and are excluded from GitHub using `.gitignore`.

In [ ]:
parquet_out = output_dir / "Cleaned_yellow_tripdata_2026-08.parquet"
csv_out = output_dir / "Cleaned_yellow_tripdata_2026-08.csv"

clean_df.to_parquet(parquet_out, index=False)
# CSV is optional because it is much larger and loses some dtype information.
# Uncomment only if you need it for Power BI:
# clean_df.to_csv(csv_out, index=False)

print("Saved:", parquet_out)
print("Rows saved:", len(clean_df))

## Cleaning decisions / limitations to document

- Filtering is based on **pickup month (August 2026)**; some valid drop-offs may be in September.
- Missing passenger counts stay missing; we do not invent passenger counts.
- We retained available fee and rate columns, even when partially missing.
- Trips with nonpositive distance, fare, recorded total or duration were excluded from the main **completed positive-fare trip** analysis, not declared universally erroneous.
- 1st/99th-percentile deletion was **not** applied automatically.
- `total_amount` is used as a proxy for recorded passenger charges, **not driver/company profit**.
- No actual surge pricing is applied to TLC records. It is a later hypothetical modeling exercise.

**Next notebook:** `02_eda_analysis_using_SQL_&_Python.ipynb` will calculate hourly demand, charge patterns, weekday/weekend comparisons, pickup-zone results, and pricing simulations from the cleaned Parquet output.